In [1]:
using DataFrames

### Joining, Grouping, Split-Apply-Combine

In [2]:
name = DataFrame(id=[1, 2, 3], name=["John Doe", "Jane Doe", "Jean Doe"])
job = DataFrame(id=[1, 2, 4], 
                job=["butcher", "baker", "candlestick maker"])
innerjoin(name, job, on=:id)

,id,name,job
,Int64,String,String
1,1,John Doe,butcher
2,2,Jane Doe,baker


In [3]:
job2 = DataFrame(identifier=[1, 2, 4], 
                 job=["butcher", "baker", "candlestick maker"])
innerjoin(name, job2, on=:id => :identifier)

,id,name,job
,Int64,String,String
1,1,John Doe,butcher
2,2,Jane Doe,baker


In [4]:
innerjoin(name, job2, on=[:id => :identifier])

,id,name,job
,Int64,String,String
1,1,John Doe,butcher
2,2,Jane Doe,baker


In [5]:
leftjoin(name, job, on=:id)

,id,name,job
,Int64,String,String?
1,1,John Doe,butcher
2,2,Jane Doe,baker
3,3,Jean Doe,missing


In [6]:
outerjoin(name, job, on=:id)

,id,name,job
,Int64,String?,String?
1,1,John Doe,butcher
2,2,Jane Doe,baker
3,3,Jean Doe,missing
4,4,missing,candlestick maker


In [7]:
antijoin(name, job, on=:id)

,id,name
,Int64,String
1,3,Jean Doe


In [3]:
semijoin(name, job, on=:id) # SELECT name FROM name INNER JOIN job ON id ?

,id,name
,Int64,String
1,1,John Doe
2,2,Jane Doe


In [4]:
df1 = DataFrame(X=1:3)

,X
,Int64
1,1
2,2
3,3


In [5]:
df2 = DataFrame(Y=["a", "b"])

,Y
,String
1,a
2,b


In [6]:
crossjoin(df1, df2)

,X,Y
,Int64,String
1,1,a
2,1,b
3,2,a
4,2,b
5,3,a
6,3,b


In [7]:
df = DataFrame(a=1:3, b=4:6)

,a,b
,Int64,Int64
1,1,4
2,2,5
3,3,6


In [8]:
combine(df, :a => sum, nrow)

,a_sum,nrow
,Int64,Int64
1,6,3


In [9]:
df = DataFrame(a=repeat([1, 2, 3, 4], outer=2),
               b=repeat([2, 1], outer=[4]),
               c=1:8)

,a,b,c
,Int64,Int64,Int64
1,1,2,1
2,2,1,2
3,3,2,3
4,4,1,4
5,1,2,5
6,2,1,6
7,3,2,7
8,4,1,8


In [11]:
gd = groupby(df, :a)
combine(gd, :c => sum, nrow)

,a,c_sum,nrow
,Int64,Int64,Int64
1,1,6,2
2,2,8,2
3,3,10,2
4,4,12,2


In [12]:
combine(gd, :c => sum, nrow, ungroup=false)

,a,c_sum,nrow
,Int64,Int64,Int64
1,1,6,2
,a,c_sum,nrow
,Int64,Int64,Int64
1,4,12,2


In [13]:
combine(sdf -> sum(sdf.c), gd) # slower variant

,a,x1
,Int64,Int64
1,1,6
2,2,8
3,3,10
4,4,12


In [15]:
combine(gd) do d # slower variant
    sum(d.c)
end

,a,x1
,Int64,Int64
1,1,6
2,2,8
3,3,10
4,4,12


In [16]:
combine(gd, [:b, :c] .=> sum)

,a,b_sum,c_sum
,Int64,Int64,Int64
1,1,4,6
2,2,2,8
3,3,4,10
4,4,2,12


In [17]:
combine(gd) do sdf
    sdf.c[1] != 1 ? sdf : DataFrame()
end

,a,b,c
,Int64,Int64,Int64
1,2,1,2
2,2,1,6
3,3,2,3
4,3,2,7
5,4,1,4
6,4,1,8


In [18]:
combine(gd, :b => :b1, :c => :c1, [:b, :c] => +, keepkeys=false)

,b1,c1,b_c_+
,Int64,Int64,Int64
1,2,1,3
2,2,5,7
3,1,2,3
4,1,6,7
5,2,3,5
6,2,7,9
7,1,4,5
8,1,8,9


In [19]:
combine(gd, :b, :c => sum)

,a,b,c_sum
,Int64,Int64,Int64
1,1,2,6
2,1,2,6
3,2,1,8
4,2,1,8
5,3,2,10
6,3,2,10
7,4,1,12
8,4,1,12


In [20]:
combine(gd, [:b, :c] .=> Ref)

,a,b_Ref,c_Ref
,Int64,SubArra…,SubArra…
1,1,"[2, 2]","[1, 5]"
2,2,"[1, 1]","[2, 6]"
3,3,"[2, 2]","[3, 7]"
4,4,"[1, 1]","[4, 8]"


In [21]:
combine(gd, AsTable(:) => Ref)

,a,a_b_c_Ref
,Int64,NamedTu…
1,1,"(a = [1, 1], b = [2, 2], c = [1, 5])"
2,2,"(a = [2, 2], b = [1, 1], c = [2, 6])"
3,3,"(a = [3, 3], b = [2, 2], c = [3, 7])"
4,4,"(a = [4, 4], b = [1, 1], c = [4, 8])"


In [22]:
combine(gd, :, AsTable(Not(:a)) => sum)

,a,b,c,b_c_sum
,Int64,Int64,Int64,Int64
1,1,2,1,3
2,1,2,5,7
3,2,1,2,3
4,2,1,6,7
5,3,2,3,5
6,3,2,7,9
7,4,1,4,5
8,4,1,8,9


In [3]:
df = DataFrame(a=repeat([1, 2, 3, 4], outer=[2]),
               b=repeat([2, 1], outer=[4]),
               c=1:8)

,a,b,c
,Int64,Int64,Int64
1,1,2,1
2,2,1,2
3,3,2,3
4,4,1,4
5,1,2,5
6,2,1,6
7,3,2,7
8,4,1,8


In [4]:
gd = groupby(df, :a)

,a,b,c
,Int64,Int64,Int64
1,1,2,1
2,1,2,5
,a,b,c
,Int64,Int64,Int64
1,4,1,4
2,4,1,8


In [5]:
gd[2]

,a,b,c
,Int64,Int64,Int64
1,2,1,2
2,2,1,6


In [6]:
last(gd)

,a,b,c
,Int64,Int64,Int64
1,4,1,4
2,4,1,8


In [7]:
gd[(a=3,)]

,a,b,c
,Int64,Int64,Int64
1,3,2,3
2,3,2,7


In [8]:
gd[(3, )]

,a,b,c
,Int64,Int64,Int64
1,3,2,3
2,3,2,7


In [9]:
k = first(keys(gd))

GroupKey: (a = 1,)

In [10]:
gd[k]

,a,b,c
,Int64,Int64,Int64
1,1,2,1
2,1,2,5


In [11]:
for g in gd
    println(g)
end

2×3 SubDataFrame
│ Row │ a     │ b     │ c     │
│     │ Int64 │ Int64 │ Int64 │
├─────┼───────┼───────┼───────┤
│ 1   │ 1     │ 2     │ 1     │
│ 2   │ 1     │ 2     │ 5     │
2×3 SubDataFrame
│ Row │ a     │ b     │ c     │
│     │ Int64 │ Int64 │ Int64 │
├─────┼───────┼───────┼───────┤
│ 1   │ 2     │ 1     │ 2     │
│ 2   │ 2     │ 1     │ 6     │
2×3 SubDataFrame
│ Row │ a     │ b     │ c     │
│     │ Int64 │ Int64 │ Int64 │
├─────┼───────┼───────┼───────┤
│ 1   │ 3     │ 2     │ 3     │
│ 2   │ 3     │ 2     │ 7     │
2×3 SubDataFrame
│ Row │ a     │ b     │ c     │
│     │ Int64 │ Int64 │ Int64 │
├─────┼───────┼───────┼───────┤
│ 1   │ 4     │ 1     │ 4     │
│ 2   │ 4     │ 1     │ 8     │


In [12]:
df = DataFrame(a=repeat([:foo, :bar, :baz], outer=[4]),
               b=repeat([2, 1], outer=6),
               c=1:12)

,a,b,c
,Symbol,Int64,Int64
1,foo,2,1
2,bar,1,2
3,baz,2,3
4,foo,1,4
5,bar,2,5
6,baz,1,6
7,foo,2,7
8,bar,1,8
9,baz,2,9


In [13]:
gd = groupby(df, [:a, :b])

,a,b,c
,Symbol,Int64,Int64
1,foo,2,1
2,foo,2,7
,a,b,c
,Symbol,Int64,Int64
1,baz,1,6
2,baz,1,12


In [14]:
keys(gd)

6-element DataFrames.GroupKeys{GroupedDataFrame{DataFrame}}:
 GroupKey: (a = :foo, b = 2)
 GroupKey: (a = :bar, b = 1)
 GroupKey: (a = :baz, b = 2)
 GroupKey: (a = :foo, b = 1)
 GroupKey: (a = :bar, b = 2)
 GroupKey: (a = :baz, b = 1)

In [15]:
k = keys(gd)[1]

GroupKey: (a = :foo, b = 2)

In [16]:
keys(k)

2-element Array{Symbol,1}:
 :a
 :b

In [17]:
values(k)

(:foo, 2)

In [18]:
k.a

:foo

In [19]:
k[:a]

:foo

In [20]:
k[1]

:foo

In [21]:
gd[k]

,a,b,c
,Symbol,Int64,Int64
1,foo,2,1
2,foo,2,7
